<a href="https://colab.research.google.com/github/Andrian0s/ML4NLP1-2026-Tutorial-Notebooks/blob/main/exercises/ex5/ex5_part2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exercise 5: LLM Prompting and Prompt Engineering Part 2

In part 2, we experiment with prompting instruction-tuned Large Language Models (LLMs), and evaluate their performance on a linguistic annotation task involving structured outputs.

The goal of this assignment is to gain some experience working with instruction-tuned LLMs. To this end, you will learn how to

- query an instruction-tuned LLM with default chat templates (`Llama-3.2-3B-Instruct`)
- parse LLM outputs for structured responses using JSON and `Pydantic`
- implement error handling for edge cases where the model fails to output the expected data format.
- compare the instruction-tuned model with the base model from part 1 on the same task.

The task we use for this purpose is a simple Tokenization and Part-of-Speech tagging task using data taken from Universal Dependencies.

To facilitate working with LLMs, we will again use the Unsloth library. Note that Unsloth provides both freeware and closed-source proprietary software. For our purposes, the freeware is sufficient! For more information on Unsloth, see the docs [here](https://github.com/unslothai/unsloth).

This notebook is adapted from [this example](https://colab.research.google.com/drive/1T5-zKWM_5OD21QHwXHiV9ixTRR7k3iB9?usp=sharing) by Unsloth.


Throughout the notebook, there are questions that you should address in your report. These are marked with 📝❓ and are collected again in the lab report section at the end of the notebook.

### NOTE: Expected execution times
We have provided expected execution times throughout the notebook as a guide. These are intended to be approximate, but should give you some idea for what to expect. If your runtimes far exceed these expected execution times, you may want to consider modifying your approach. These are denoted with ⌛ .

### NOTE: GPU Usage
It is expected that you load the model onto a GPU for inference. For other parts of the code, such as data preparation, a GPU is not necessary. To avoid waiting for resources unnecessarily, we recommend doing as much as you can on a CPU instance and change the runtime type as necessary. We've highlighted the cells that need a GPU with ⚡

## Use of generative AI

AI tools may be used in this exercise to help you explain code, debug, improve documentation, and explore alternatives. Any use must be documented in the *Declaration on the use of AI* at the end of this notebook (which tools were used and for which purposes). You remain responsible for understanding, checking, and testing everything you submit, and you must be able to explain any submitted text, code, analyses, and AI-assisted contributions. See the [CL guidelines on academic integrity and AI use](https://www.cl.uzh.ch/en/Studies/Services/academic-integrity.html).

## 1) Installing dependencies

In [ ]:
%%capture
!pip install levenshtein
!pip install unsloth==2026.11.4 unsloth_zoo==2026.11.4

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("No GPU found. This exercise needs an NVIDIA GPU: in Colab, select one at 'Runtime' → 'Change runtime type'.")

## 2) Model Loading

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# Note, here we specify the instruction-tuned version of Llama-3.2-3B
model_name = "unsloth/Llama-3.2-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)


FastLanguageModel.for_inference(model) # Enable native 2x faster inference

## 3) Data Loading and Preparation

In [ ]:
# load data
import random
import pandas as pd
from transformers import set_seed

seed = 42

set_seed(seed) # seeds Python, NumPy and PyTorch (CPU and GPU)

dataset_url = "https://raw.githubusercontent.com/tannonk/prompting_exercise/refs/heads/main/data/en_ewt-ud-dev-pos.json"
df = pd.read_json(dataset_url, lines=True)

# For each input sentence, we'll build the target as a list of dictionaries containing keys for the token and pos tag. This is what we want our LLM annotator to predict.
df['target'] = df.apply(lambda x: [{'token': token, 'pos': pos} for token, pos in zip(x['tokens'].split(), x['upos'].split())], axis=1)

# We'll sample 100 items for testing purposes
test_data = df.sample(n=100, random_state=seed)
train_data = df.drop(test_data.index)

print(f"Train data: {len(train_data)}")
print(f"Test data: {len(test_data)}")

test_data.head()


### TODO: Inspect and describe the data

📝❓ What are the fields and their corresponding values in the dataframe?

📝❓ What is the difference between `upos` and `xpos`?

📝❓ What is the distribution of `upos` labels in the `test_data`?

### TODO: Define the basic `PromptTemplate`

Note, you can reuse the solution from part 1 of this exercise here.

In [ ]:
# TODO

## 3.2 ChatTemplates

Instruction-tuned models are typically finetuned using a predefined `ChatTemplate`.
This means that when using them for inference, it is important that we use the correct `ChatTemplate` in order to avoid "confusing" the model.

You can find more information about model `ChatTemplates` for Huggingface models [here](https://huggingface.co/docs/transformers/en/chat_templating).


In [ ]:
from unsloth.chat_templates import get_chat_template

# load the chat_template from unsloth (note, the logic is similar when using native Huggingface, but here we're using Unsloth!)
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "llama-3.1", # for Llama-3.1 and Llama-3.2 models
)

# Inspect the template (note, it looks more complicated than it is!)
print(tokenizer.chat_template)


### TODO: Prepare your inputs using the `ChatTemplate` for the model.

Note, you should be able to drop your custom `PromptTemplate` string into the model's default `ChatTemplate`.


In [ ]:
# TODO

## 4) Inference Pipeline

### TODO: Define a function to run inference efficiently with an LLM

Note, you can use the same inference function from part 1 of this exercise here, with one change: the prompts produced by the chat template already start with the beginning-of-sequence token `<|begin_of_text|>`, which the tokenizer would otherwise add a second time. Tokenise such prompts with `add_special_tokens=False`.

In [ ]:
# Set up our inference pipeline for generation

# We'll set some default generation args that we'll pass to our inference function
# Following best practices, we'll use Pydantic class which helps with validation.
from typing import Optional
from pydantic import BaseModel

class Generation_Args(BaseModel):
    max_new_tokens: int
    temperature: float
    top_k: int
    top_p: float
    repetition_penalty: float
    do_sample: bool
    min_p: Optional[float]
    num_return_sequences: int
    use_cache: bool

# Here are some default generation args
# We use greedy decoding (do_sample = False): at each step, the model picks the most probable next token.
# This makes the outputs deterministic, so that differences in accuracy come from the prompts and not from random sampling.
# temperature, top_k, top_p and min_p only take effect if you set do_sample = True.
generation_args = Generation_Args(
    max_new_tokens = 1024, # note, for this task, we're setting the max_new_tokens to be more appropriate
    temperature = 1.0,
    top_k = 50,
    top_p = 1.0,
    repetition_penalty = 1.0,
    do_sample = False,
    use_cache = True,
    min_p = None,
    num_return_sequences = 1
)


def run_batched_inference(prompts, model, tokenizer, batch_size=10, generation_args=generation_args):
    """
    Runs batched inference on a list of prompts using a given model and tokenizer.

    Set the batch_size to control the number of prompts processed in each batch.
    Depending on the length of your prompts and model size the batch size may need to be adjusted.

    Args:
        prompts (list[str]): List of prompts that are passed to the model
        model (): The model used for generation
        tokenizer (): The tokenizer used for encoding and decoding the prompts
        batch_size (int): Number of prompts to run batched inference on.

    Returns:
        List[str] containing generated outputs.
    """
    tokenizer.padding_side = "left"

    outputs = []

    # TODO: implement the logic for efficient inference with LLM

    return outputs

## 4.1) Run Inference

### TODO: run inference!

⌛ 10-20 mins

⚡ GPU

In [ ]:
# TODO: inference

## 5) Structured output validation

LLMs output text. But in practice, we often want structured data that we can process further with other automatic processes.

For this purpose, JSON is a good target data structure.


### TODO: Define a processing pipeline that extracts and validates the JSON response from the LLM.

Hint: For this you should use a combination of [`Regex`](https://www.w3schools.com/python/python_regex.asp) and [`Pydantic`](https://docs.pydantic.dev/latest/).

The output should be a valid json object with the following structure:

```json
[
    {"token": "there", "pos": "DET"}, # each dict contains a token and its corresponding POS-Tag.
    {"token": "is", "pos": "VERB"},
    {"token": "no", "pos": "ADJ"},
    {"token": "companion", "pos": "NOUN"},
    {"token": "quite", "pos": "ADV"},
    {"token": "so", "pos": "ADV"},
    {"token": "devoted", "pos": "ADV"},
    {"token": "so", "pos": "ADV"},
    ...
]
```

In [ ]:
# TODO

## 6) *Evaluation*

We align the predicted tokens with the target tokens and compute precision, recall and F1 over all test sentences, once for the tokens and once for the POS tags. A predicted token is correct if it is aligned with an identical target token; its POS tag is correct if, in addition, the tags are identical.

- **Precision** is the share of the predicted tokens that are correct, so extra tokens lower it.
- **Recall** is the share of the target tokens that are predicted correctly, so missing tokens and outputs that could not be parsed lower it.

In [ ]:
# Below is some boilerplate evaluation code. You should not need to make any changes here.

import Levenshtein
import numpy as np
from typing import List, Dict, Optional

def evaluate_instance(target: List[Dict], prediction: Optional[List[Dict]]):
    """
    Compares the predicted tokens and POS tags of one sentence with the target.

    The predicted tokens are aligned with the target tokens using Levenshtein.opcodes(). A predicted token is correct
    if it is aligned with an identical target token. Its POS tag is correct if, in addition, the tags are identical.

    Args:
        target (List[Dict]): A list of dictionaries representing the target tokens and POS tags.
        prediction (List[Dict] or None): A list of dictionaries representing the predicted tokens and POS tags.

    Returns:
        dict: The number of target tokens, predicted tokens, correct tokens and correct POS tags.
    """

    target_tokens = [item['token'] for item in target]
    target_pos = [item['pos'] for item in target]

    # If there is no prediction, nothing is predicted and all target tokens are missed
    if prediction is None:
        return {'target': len(target_tokens), 'predicted': 0, 'correct tokens': 0, 'correct POS': 0}

    pred_tokens = [item['token'] for item in prediction]
    pred_pos = [item['pos'] for item in prediction]

    correct_tokens = 0
    correct_pos = 0

    # Get alignment operations between the target and predicted tokens using Levenshtein.opcodes()
    # "equal" blocks contain identical tokens; "replace", "insert" and "delete" blocks are tokenisation errors
    for tag, i1, i2, j1, j2 in Levenshtein.opcodes(target_tokens, pred_tokens):
        if tag == 'equal':
            correct_tokens += i2 - i1
            correct_pos += sum(t == p for t, p in zip(target_pos[i1:i2], pred_pos[j1:j2]))

    return {'target': len(target_tokens), 'predicted': len(pred_tokens),
            'correct tokens': correct_tokens, 'correct POS': correct_pos}

def precision_recall_f1(correct, predicted, target):
    precision = correct / predicted if predicted else 0
    recall = correct / target if target else 0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0
    return precision, recall, f1

def get_results(test_data: pd.DataFrame, processed_outputs: List[Optional[List[Dict]]]):
    """
    Returns precision, recall and F1 for tokenisation and POS tagging, computed over all sentences.
    The i-th output is compared with the i-th sentence of test_data.
    """
    counts = pd.DataFrame([evaluate_instance(test_data.iloc[i]['target'], processed_outputs[i])
                           for i in range(len(processed_outputs))]).sum()

    token_p, token_r, token_f1 = precision_recall_f1(counts['correct tokens'], counts['predicted'], counts['target'])
    pos_p, pos_r, pos_f1 = precision_recall_f1(counts['correct POS'], counts['predicted'], counts['target'])

    return pd.Series({'Token Precision': token_p, 'Token Recall': token_r, 'Token F1': token_f1,
                      'POS Precision': pos_p, 'POS Recall': pos_r, 'POS F1': pos_f1})

In [ ]:
# To get the results, you should be able to pass your test_data DataFrame and the processed_outputs from above...
get_results(test_data, processed_outputs)

## 7) Manipulating the system prompt

The system prompt is part of the `ChatTemplate` that can help to steer the model.


### TODO: Customise the system prompt for the intended task and re-run inference

Note, this is an experiment. You should try a few different system prompts and report the resulting performance in your report.


📝❓ What was the best system prompt you considered?

📝❓ Were you able to improve the performance by manipulating the system prompt? Please discuss.

⌛ 10-20 mins (per experiment run)

⚡ GPU

In [ ]:
# TODO: inference

## 8) Base model vs. instruction-tuned model

`Llama-3.2-3B-Instruct` was created from the base model `Llama-3.2-3B` that you used in part 1. The base model was trained further on many different tasks phrased as instructions together with the desired responses, formatted with the chat template (instruction tuning), and then on human preferences. Let's see what this changes for our task.

### TODO: Run the base model on the same task and compare

Load the base model and run it on the same test sentences with the prompt from your `PromptTemplate`. The base model was not trained with a chat template, so pass the prompt as plain text. Then process and evaluate the outputs with the same pipeline as above.

Note, the base model may keep generating until it reaches `max_new_tokens`. To save time, you can run this comparison on a subset of the test data (e.g. `test_data.head(20)`) and compare it with the instruction-tuned model on the same subset.

📝❓ How do the outputs of the base model differ from those of the instruction-tuned model (output format, number of valid outputs, precision, recall and F1)?

📝❓ In part 1, you finetuned the base model with LoRA on a single task. How is this different from the instruction tuning that turned `Llama-3.2-3B` into `Llama-3.2-3B-Instruct`? Use your results on the out-of-domain names from part 1 in your answer.

⌛ 5-20 mins

⚡ GPU

In [ ]:
# Load the base model from part 1 under a new name, so that the instruction-tuned model stays available
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-3B-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(base_model) # Enable native 2x faster inference

# TODO: run inference with the base model, then process and evaluate the outputs

---

## 9) Lab report

📝❓ Write your lab report here, addressing all questions in the notebook. For convenience, the questions are collected below; please keep your answers clearly marked.

### Questions

**3) Data**

- 📝❓ What are the fields and their corresponding values in the dataframe?
- 📝❓ What is the difference between `upos` and `xpos`?
- 📝❓ What is the distribution of `upos` labels in the `test_data`?

**7) System prompt**

- 📝❓ What was the best system prompt you considered?
- 📝❓ Were you able to improve the performance by manipulating the system prompt? Please discuss.

**8) Base model vs. instruction-tuned model**

- 📝❓ How do the outputs of the base model differ from those of the instruction-tuned model (output format, number of valid outputs, precision, recall and F1)?
- 📝❓ In part 1, you finetuned the base model with LoRA on a single task. How is this different from the instruction tuning that turned `Llama-3.2-3B` into `Llama-3.2-3B-Instruct`? Use your results on the out-of-domain names from part 1 in your answer.

## Declarations

Please fill in both declarations below by editing this cell (double-click it). Mark the applicable option by replacing ☐ with ☒ (copy the symbol from here: ☒) and complete the text field where required. Your reviewers will check that both declarations are present and completed.

### Contribution declaration

By submitting this work, the group declares that the contributions of the group members were:

☐ approximately equal

☐ not approximately equal

If the contributions were not approximately equal, no individual group member needs to be identified.

### Declaration on the use of AI

The group declares that any use of AI-based tools in preparing this work complied with the rules specified for this exercise.

☐ No AI-based tools were used.

☐ AI-based tools were used in accordance with the rules of the exercise.

If AI-based tools were used, briefly state the tool(s) used and the purpose(s) for which they were used: